In [1]:
%matplotlib inline
# Cell 1
import warnings
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from IPython.display import Image as IPImage

from scripts.shared.db_utils import db_connect
from scripts.shared import db_utils
import app.db.seasonality as sim_mod
from app.db.seasonality import load_similarity_index, LENS_REGISTRY

ROOT = Path(db_utils.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'edop' / 'demo'
OUT.mkdir(parents=True, exist_ok=True)

print('imports ok')
print(f'output → {OUT}')

imports ok
output → /Users/karlg/Documents/repos/_edops/output/edop/demo


In [2]:
# Cell 2  — Connect + load similarity index
# Replicates server startup: loads monthly arrays, computes derived vars,
# builds per-lens state (Euclidean z-matrix or Mahalanobis VI).
# Takes ~5–10 s on first run.

conn = db_connect()
load_similarity_index(conn)

HYBAS_IDS = sim_mod._HYBAS_IDS   # (N,) int64
LENS_STATE = sim_mod._LENS_STATE  # {lens_id: state_dict}
N = len(HYBAS_IDS)

print(f'Index loaded: {N:,} basins, {len(LENS_STATE)} active lenses')
for lid, s in LENS_STATE.items():
    n_valid = int((~np.isnan(s['X_raw']).any(axis=1)).sum())
    print(f'  {lid:<20}  metric={s["metric"]}  vars={s["variables"]}  valid={n_valid:,}')

Index loaded: 16,397 basins, 3 active lenses
  climate.precip        metric=euclidean  vars=['pre_mm_syr', 'pre_concentration']  valid=16,338
  climate.temp          metric=mahalanobis  vars=['tmp_dc_syr', 'tmp_seas_amp', 'tmp_concentration']  valid=16,397
  climate.phase         metric=euclidean  vars=['pre_concentration', 'seas_phase_offset']  valid=16,338


In [3]:
# Cell 3  — Helpers: full distance array + anchor lookup

def full_distances(lens_id: str, query_hybas_id: int) -> np.ndarray:
    """Return (N,) distance array from query basin to every basin under lens_id.

    NaN-masked basins and the query basin itself get np.inf.
    """
    state = LENS_STATE[lens_id]
    hits  = np.where(HYBAS_IDS == int(query_hybas_id))[0]
    if len(hits) == 0:
        raise ValueError(f'hybas_id {query_hybas_id} not in index')
    q = int(hits[0])

    X_raw = state['X_raw']
    if state['metric'] == 'euclidean':
        Xz   = state['Xz']
        diff = Xz - Xz[q]
        dist = np.sqrt(np.nansum(diff ** 2, axis=1))
        dist[np.isnan(Xz).any(axis=1)] = np.inf
    else:
        mask = state['mask']
        diff = X_raw - X_raw[q]
        d2   = np.einsum('ij,jk,ik->i', diff, state['VI'], diff)
        dist = np.where(mask & (d2 >= 0), np.sqrt(d2), np.inf)

    dist[q] = np.inf
    return dist


def anchor_hybas(lat: float, lon: float) -> int:
    """Return the L06 hybas_id containing (lat, lon)."""
    sql = f"""
        SELECT hybas_id FROM public.basin06
        WHERE ST_Within(ST_SetSRID(ST_MakePoint({lon},{lat}),4326), geom)
        ORDER BY ST_Area(geom::geography) ASC LIMIT 1
    """
    return int(pd.read_sql(sql, conn).iloc[0, 0])


def count_within(dist: np.ndarray, radius: float) -> int:
    """Count finite-distance basins within radius."""
    return int(np.sum((dist < np.inf) & (dist <= radius)))


# Validation anchors — hybas_ids established in WO7a
ANCHORS = {
    'SF (phase rare)':        7060013180,
    'Timbuktu (phase common)': 1060551560,
    'Rome (precip)':          2060015610,
    'London (temp maritime)': 2060053790,
    'Tbilisi (temp continental)': 2060616700,
}
print('Anchors:')
for name, hid in ANCHORS.items():
    print(f'  {name:<30} hybas_id={hid}')

Anchors:
  SF (phase rare)                hybas_id=7060013180
  Timbuktu (phase common)        hybas_id=1060551560
  Rome (precip)                  hybas_id=2060015610
  London (temp maritime)         hybas_id=2060053790
  Tbilisi (temp continental)     hybas_id=2060616700


In [4]:
# Cell 4  — climate.phase CDFs: SF (rare Mediterranean) vs Timbuktu (common monsoon)
#
# Expected: SF has a sharp elbow then a gap; Timbuktu has a gentle slope.
# That contrast IS the finding — rare vs. common climate type prevalence.

LENS = 'climate.phase'
ANCHORS_PHASE = {'SF': ANCHORS['SF (phase rare)'], 'Timbuktu': ANCHORS['Timbuktu (phase common)']}
COLORS = {'SF': '#2166ac', 'Timbuktu': '#d6604d'}

dist_phase = {name: full_distances(LENS, hid) for name, hid in ANCHORS_PHASE.items()}

# CDF plot
fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

for name, dist in dist_phase.items():
    finite = np.sort(dist[dist < np.inf])
    cdf    = np.arange(1, len(finite) + 1) / len(finite)
    ax.plot(finite, cdf, color=COLORS[name], lw=1.8, label=name)

ax.set_xlabel('Distance (normalized Euclidean, z-score space)', color='black', fontsize=11)
ax.set_ylabel('Cumulative fraction of valid basins', color='black', fontsize=11)
ax.set_title('climate.phase CDF — SF vs Timbuktu', color='black', fontsize=12)
ax.set_xlim(0, 3)
ax.tick_params(colors='black')
for sp in ax.spines.values(): sp.set_edgecolor('black')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, color='gray')

fig.tight_layout()
fig.savefig(OUT / 'wo7b_phase_cdf.png', facecolor='white', dpi=120)
plt.close(fig)
display(IPImage(str(OUT / 'wo7b_phase_cdf.png')))

# Count table at candidate radii
RADII_PHASE = [0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0]
rows = []
for r in RADII_PHASE:
    row = {'radius': r}
    for name, dist in dist_phase.items():
        row[name] = count_within(dist, r)
    rows.append(row)
df_phase = pd.DataFrame(rows)
print('\nclimate.phase — basin count within radius:')
print(df_phase.to_string(index=False))


climate.phase — basin count within radius:
 radius   SF  Timbuktu
   0.10   34        74
   0.20  119       251
   0.30  216       436
   0.50  541       784
   0.75 1059      1548
   1.00 1600      2409
   1.50 2684      4206
   2.00 3867      6087


In [5]:
# Cell 5  — climate.precip CDFs: Rome (Mediterranean) vs Timbuktu (Sahel)

LENS = 'climate.precip'
ANCHORS_PRECIP = {'Rome': ANCHORS['Rome (precip)'], 'Timbuktu': ANCHORS['Timbuktu (phase common)']}
COLORS_P = {'Rome': '#1a9641', 'Timbuktu': '#d6604d'}

dist_precip = {name: full_distances(LENS, hid) for name, hid in ANCHORS_PRECIP.items()}

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

for name, dist in dist_precip.items():
    finite = np.sort(dist[dist < np.inf])
    cdf    = np.arange(1, len(finite) + 1) / len(finite)
    ax.plot(finite, cdf, color=COLORS_P[name], lw=1.8, label=name)

ax.set_xlabel('Distance (normalized Euclidean, z-score space)', color='black', fontsize=11)
ax.set_ylabel('Cumulative fraction of valid basins', color='black', fontsize=11)
ax.set_title('climate.precip CDF — Rome vs Timbuktu', color='black', fontsize=12)
ax.set_xlim(0, 3)
ax.tick_params(colors='black')
for sp in ax.spines.values(): sp.set_edgecolor('black')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, color='gray')

fig.tight_layout()
fig.savefig(OUT / 'wo7b_precip_cdf.png', facecolor='white', dpi=120)
plt.close(fig)
display(IPImage(str(OUT / 'wo7b_precip_cdf.png')))

RADII_PRECIP = [0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0]
rows = []
for r in RADII_PRECIP:
    row = {'radius': r}
    for name, dist in dist_precip.items():
        row[name] = count_within(dist, r)
    rows.append(row)
df_precip = pd.DataFrame(rows)
print('\nclimate.precip — basin count within radius:')
print(df_precip.to_string(index=False))


climate.precip — basin count within radius:
 radius  Rome  Timbuktu
   0.10    72        71
   0.20   246       239
   0.30   541       450
   0.50  1521       750
   0.75  3052      1378
   1.00  4877      2285
   1.50  7961      4100
   2.00 10366      6202


In [6]:
# Cell 6  — climate.temp CDFs: London (maritime) vs Tbilisi (continental)
#
# Mahalanobis distances: scale differs from Euclidean lenses.
# Expect larger numeric values; x-axis range adjusted accordingly.

LENS = 'climate.temp'
ANCHORS_TEMP = {'London': ANCHORS['London (temp maritime)'], 'Tbilisi': ANCHORS['Tbilisi (temp continental)']}
COLORS_T = {'London': '#4d9de0', 'Tbilisi': '#e15554'}

dist_temp = {name: full_distances(LENS, hid) for name, hid in ANCHORS_TEMP.items()}

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

for name, dist in dist_temp.items():
    finite = np.sort(dist[dist < np.inf])
    cdf    = np.arange(1, len(finite) + 1) / len(finite)
    ax.plot(finite, cdf, color=COLORS_T[name], lw=1.8, label=name)
    p10 = finite[int(0.10 * len(finite))]
    p25 = finite[int(0.25 * len(finite))]
    print(f'{name}: 10th pctile dist={p10:.3f} ({count_within(dist, p10)} basins)  '
          f'25th pctile dist={p25:.3f} ({count_within(dist, p25)} basins)')

ax.set_xlabel('Distance (Mahalanobis)', color='black', fontsize=11)
ax.set_ylabel('Cumulative fraction of valid basins', color='black', fontsize=11)
ax.set_title('climate.temp CDF — London vs Tbilisi', color='black', fontsize=12)
ax.tick_params(colors='black')
for sp in ax.spines.values(): sp.set_edgecolor('black')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, color='gray')

fig.tight_layout()
fig.savefig(OUT / 'wo7b_temp_cdf.png', facecolor='white', dpi=120)
plt.close(fig)
display(IPImage(str(OUT / 'wo7b_temp_cdf.png')))

# Use wider candidate range for Mahalanobis
RADII_TEMP = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0]
rows = []
for r in RADII_TEMP:
    row = {'radius': r}
    for name, dist in dist_temp.items():
        row[name] = count_within(dist, r)
    rows.append(row)
df_temp = pd.DataFrame(rows)
print('\nclimate.temp — basin count within radius:')
print(df_temp.to_string(index=False))


climate.temp — basin count within radius:
 radius  London  Tbilisi
   0.25      52       61
   0.50     234      339
   0.75     550      852
   1.00    1094     1672
   1.50    3242     5272
   2.00    6708    10939
   3.00   13759    15610
   4.00   15974    16230


In [7]:
# Cell 7  — All-anchor count table per lens: cross-lens comparison
#
# For each lens, show the count at each candidate radius for ALL validation
# anchors together. This makes it easy to read off strict/moderate/loose.
# Fill in the settled radii below after inspecting cells 4–6.

print('=== climate.phase ===')
print(df_phase.to_string(index=False))

print('\n=== climate.precip ===')
print(df_precip.to_string(index=False))

print('\n=== climate.temp ===')
print(df_temp.to_string(index=False))

print("""
Reading guide
-------------
strict  — tight known cluster for the rare-type anchor (SF / Rome); a few dozen basins.
moderate — working default; honest neighborhood for both rare and common types.
loose   — inclusive; lets common-type (Timbuktu) bloom to its full extent.

Note: the same radius gives very different counts across anchors. That's correct —
it's the prevalence signal WO7b is designed to reveal.
""")

=== climate.phase ===
 radius   SF  Timbuktu
   0.10   34        74
   0.20  119       251
   0.30  216       436
   0.50  541       784
   0.75 1059      1548
   1.00 1600      2409
   1.50 2684      4206
   2.00 3867      6087

=== climate.precip ===
 radius  Rome  Timbuktu
   0.10    72        71
   0.20   246       239
   0.30   541       450
   0.50  1521       750
   0.75  3052      1378
   1.00  4877      2285
   1.50  7961      4100
   2.00 10366      6202

=== climate.temp ===
 radius  London  Tbilisi
   0.25      52       61
   0.50     234      339
   0.75     550      852
   1.00    1094     1672
   1.50    3242     5272
   2.00    6708    10939
   3.00   13759    15610
   4.00   15974    16230

Reading guide
-------------
strict  — tight known cluster for the rare-type anchor (SF / Rome); a few dozen basins.
moderate — working default; honest neighborhood for both rare and common types.
loose   — inclusive; lets common-type (Timbuktu) bloom to its full extent.

Note: the s

In [8]:
# Cell 8  — Settled radii (fill in after inspecting cells 4–7)
#
# Replace the ??? placeholders with the radii chosen from the count tables.
# Run this cell to confirm the counts look right before recording in findings.

SETTLED = {
    'climate.phase': {
        'strict':   0.10,   # target: SF tight cluster, ~tens of basins
        'moderate': 0.30,   # default
        'loose':    0.75,   # Timbuktu full monsoon belt
    },
    'climate.precip': {
        'strict':   0.10,
        'moderate': 0.20,
        'loose':    0.50,
    },
    'climate.temp': {
        'strict':   0.25,   # Mahalanobis scale — expect larger numbers than NE lenses
        'moderate': 0.75,
        'loose':    1.50,
    },
}

# Verification: counts at settled radii for each anchor
for lens_id, thresholds in SETTLED.items():
    print(f'\n{lens_id}')
    anchors_for_lens = {
        'climate.phase':  dist_phase,
        'climate.precip': dist_precip,
        'climate.temp':   dist_temp,
    }[lens_id]
    for stringency, radius in thresholds.items():
        counts = {name: count_within(d, radius) for name, d in anchors_for_lens.items()}
        print(f'  {stringency:<10} r={radius}  →  {counts}')


climate.phase
  strict     r=0.1  →  {'SF': 34, 'Timbuktu': 74}
  moderate   r=0.3  →  {'SF': 216, 'Timbuktu': 436}
  loose      r=0.75  →  {'SF': 1059, 'Timbuktu': 1548}

climate.precip
  strict     r=0.1  →  {'Rome': 72, 'Timbuktu': 71}
  moderate   r=0.2  →  {'Rome': 246, 'Timbuktu': 239}
  loose      r=0.5  →  {'Rome': 1521, 'Timbuktu': 750}

climate.temp
  strict     r=0.25  →  {'London': 52, 'Tbilisi': 61}
  moderate   r=0.75  →  {'London': 550, 'Tbilisi': 852}
  loose      r=1.5  →  {'London': 3242, 'Tbilisi': 5272}
